In [2]:
# import time
import numpy as np
import scipy as sp
# from scipy.differentiate import derivative
# from scipy.interpolate import RegularGridInterpolator
import matplotlib.pyplot as plt
#import matplotlib.colors as colors
import matplotlib as mpl
mpl.rcParams.update(mpl.rcParamsDefault)
#import time

In [37]:
%matplotlib tk

In [3]:
# ===============================================================
# True functions
# ===============================================================

c_p = 0.43
c_beta = 833.33333333
c_gamma = 0.6


def F( x , v_B ):
  return 1 / (( x**(c_p) ) + v_B )

@np.vectorize
def f( x , v_B ):
  return sp.integrate.quad( lambda u: F( u , v_B )  , 0 , x )[0]
  
def h( x , v_B ):
  return ( x + c_beta ) * F( x , v_B ) 

def j( x , v_B ):
  val = h( x , v_B ) + ( c_gamma * f( x , v_B ) )
  return val


In [30]:
# ===============================================================
# Lookup table construction
# ===============================================================

x_bits = [12, 13, 14, 15, 16]   # How many indices for the x dimension. Anything after 15 bit shows no change in error
y_bits =  5    # How many indices for the y dimension. Anything after 5 bit shows no change in error
output_bits = 20 # Scaling of the output. No change after 20 bits
x_start = 0
y_start = 0.4
full_x_range = 512
full_y_range = .4625 - y_start

LUT = {}

for i in range(len(x_bits)):
  LUT[i] = np.zeros((2**x_bits[i], 2**y_bits))
  for Y in range( (2**y_bits) ):
    for X in range((2**x_bits[i]) ):
      LUT[i][X,Y] = int( (2**output_bits) * j( full_x_range * X/(2**x_bits[i]), full_y_range * Y/(2**y_bits) + y_start ) )

In [31]:
# ===============================================================
# Lookup table conversion to hexadecimal for use in VHDL
# ===============================================================

# full_table_area = 2**x_bits * 2**y_bits

# VHDL_LUT = np.empty(full_table_area, dtype=object)

# i = 0
# while i < full_table_area:
#   for X in range((2**x_bits) ):
#     for Y in range( (2**y_bits) ):
#       VHDL_LUT[i] = f"{i} => 0x\"{LUT[X,Y].astype(int):08X}\", -- row = {X}  col = {Y}"
#       i = i + 1

# np.savetxt("Bragg_LUT_VHDL.csv", np.vstack(VHDL_LUT), delimiter="", fmt='%s')

In [ ]:
# ===============================================================
# interpolation function
# ===============================================================

def j_int(x, v_B, li): #v_B given from 0 to 0.0625
  scaled_x = (2**x_bits[li]) * (x)/full_x_range 
  scaled_y = (2**y_bits) * (v_B)/full_y_range 
  
  index_x = min( int( scaled_x ), (2**x_bits[li])-1 )
  next_index_x = min( index_x+1 , (2**x_bits[li])-1 )  
  fraction_x = scaled_x - index_x
  
  index_y = min( int( scaled_y ), (2**y_bits)-1 )
  next_index_y = min( index_y+1 , (2**y_bits)-1 )  
  fraction_y = round(scaled_y - index_y, 20)
  
  P00, P01, P10, P11 = LUT[li][index_x, index_y], LUT[li][index_x, next_index_y], LUT[li][next_index_x, index_y], LUT[li][next_index_x, next_index_y]

  # bilinear interpolation
  #return P00 * (1 - fraction_x) * (1 - fraction_y) + P01 * (1 - fraction_x) * fraction_y + P10 * fraction_x * (1 - fraction_y) + P11 * fraction_x * fraction_y

  L1 = P00 + int(fraction_y * (P01 - P00))
  L2 = P10 + fraction_y * (P11 - P10)

  if L1 < 0:
    print("L1 negative")
  if L2 < 0:
    print("L2 negative")


  return((int(L1) + int( fraction_x * (L2 - L1) )) / 2**output_bits)

In [52]:
# ===============================================================
# VHDL output comparison
# ===============================================================

print(j_int(((52428800 / (2**20))), ((full_y_range) * (838860 / (2**20))) + y_start, 3))
print(j_int(((1048 / (2**20))), ((full_y_range) * (503316 / (2**20))) + y_start, 3))
print(j_int(((0 / (2**20))), ((full_y_range) * (0 / (2**20))) + y_start, 3))
print(j_int(((419430400 / (2**20))), ((full_y_range) * (1048575 / (2**20))) + y_start, 3))
print(j_int(((512000 / (2**20))), ((full_y_range) * (524288 / (2**20))) + y_start, 3))
print(j_int(((262144000 / (2**20))), ((full_y_range) * (800000 / (2**20))) + y_start, 3))
print(j_int(((31458 / (2**20))), ((full_y_range) * (671088 / (2**20))) + y_start, 3))

159.69209384918213
1903.2838249206543
2083.333333015442
119.84102821350098
715.4664392471313
118.83677196502686
1262.1999559402466


In [ ]:
# ===============================================================
# Graphing
# ===============================================================


# fig, ax1 = plt.subplots(1)

# x = np.linspace(30, x_start+full_x_range - 1, 1000 )
# ax1.plot( x, np.vectorize( j )( x , .4 ) - np.vectorize( j )( x , .45 ), label=f"j( x , .4 ) - j( x , .45 )" )
# ax1.legend()

# plt.show()

# fig, ax1 = plt.subplots(1)

# y = np.linspace(0.4,0.45,1000)
# ax1.plot( y, np.vectorize( j )( 150 , y ), label=f"j( 150, y )" )
# ax1.legend()

# plt.show()


# fig, [ax1, ax2, ax3] = plt.subplots(1,3)

# x = np.linspace(30, x_start+full_x_range - 1, 1000 )
# ax1.plot( x, np.vectorize( j )( x , .44 ) , label=f"j( x , .44 )" )
# ax1.legend()

# x = np.linspace(30, x_start+full_x_range - 1, 1000 )
# ax2.plot( x, np.vectorize( j_int )( x , .44), label=f"j( x , .44 )" )
# ax2.legend()

# x = np.linspace(30, x_start+full_x_range - 1, 1000 ) 
# ax3.plot( x, ((np.vectorize( j_int )( x , .44 )) - np.vectorize( j )( x , .44 )), label=f"j( x , .44 )" )
# ax3.legend()

# plt.show()


# fig, [ax1, ax2, ax3] = plt.subplots(1,3)

# x,y = np.linspace(30, x_start+full_x_range - 1, 1000 ) , np.linspace(0.4,0.45,7)
# for Y in y: ax1.plot( x, np.vectorize( j )( x , Y ) , label=f"j( x , {Y:.02f} )" )
# ax1.set_title("Exact j function")
# ax1.set_xlabel("x")
# ax1.set_ylabel('j')
# ax1.legend()

# x,y = np.linspace(30, x_start+full_x_range - 1, 1000 ) , np.linspace(y_start, y_start+full_y_range - .01, 7) 
# for Y in y: ax2.plot( x, np.vectorize( j_int )( x , Y ), label=f"j( x , {Y:.02f} )" )
# ax2.set_title("Interpolated j function")
# ax2.set_xlabel("x")
# ax2.set_ylabel('j')
# ax2.legend()

# x,y = np.linspace(30, x_start+full_x_range - 1, 1000 ) , np.linspace(y_start, y_start+full_y_range -.01, 7) 
# for Y in y: ax3.plot( x, ((np.vectorize( j_int )( x , Y )) - np.vectorize( j )( x , Y )), label=f"j( x , {Y:.02f} )" )
# ax3.set_title("Difference between interpolated and exact j function")
# ax3.set_xlabel("x")
# ax3.set_ylabel('j')
# ax3.legend()

# plt.show()

# fig, ax1 = plt.subplots(1)

# x,y = np.linspace(0, 512, 1000 ) , np.linspace(0.4,0.45,7)
# for Y in y: ax1.plot( x, np.vectorize( j )( x , Y ) , label=f"j( x , {Y:.02f} )" )
# ax1.set_title("j function")
# ax1.set_xlabel("x")
# ax1.set_ylabel('j')
# ax1.legend()

# plt.show()

# fig, [ax1, ax2, ax3] = plt.subplots(1,3)

# x,y = np.linspace(30, x_start+full_x_range - 1, 10 ) , np.linspace(y_start, y_start+full_y_range -.01, 1000)
# for X in x: ax1.plot( y, np.vectorize( j )( X , y ) , label=f"j( {X:.0f} ," + r' $v_B$)')
# ax1.set_title("Exact j function")
# ax1.set_xlabel(r'$v_B$')
# ax1.set_ylabel('j')
# ax1.legend()


# x,y = np.linspace(30, x_start+full_x_range - 1, 10 ) , np.linspace(y_start, y_start+full_y_range -.01, 1000)
# for X in x: ax2.plot( y, np.vectorize( j_int )( X , y ), label=f"j( {X:.0f} ," + r' $v_B$)')
# ax2.set_title("Interpolated j function")
# ax2.set_xlabel(r'$v_B$')
# ax2.set_ylabel('j')
# ax2.legend()

# x,y = np.linspace(30, x_start+full_x_range - 1, 10 ) , np.linspace(y_start, y_start+full_y_range - .01, 1000)
# for X in x: ax3.plot( y, ((np.vectorize( j_int )( X , y )) - np.vectorize( j )( X , y )) , label=f"j( {X:.0f} ," + r' $v_B$)')
# ax3.set_title("Difference between interpolated and exact j function")
# ax3.set_xlabel(r'$v_B$')
# ax3.set_ylabel('j difference')
# ax3.legend()

# plt.show()

# fig, ax1 = plt.subplots(1)

# x,y = np.linspace(30, x_start+full_x_range - 1, 10 ) , np.linspace(y_start, y_start+full_y_range -.01, 1000)
# for X in x: ax1.plot( y, np.vectorize( j )( X , y ) , label=f"j( {X:.0f} ," + r' $v_B$)')
# ax1.set_title("Exact j function")
# ax1.set_xlabel(r'$v_B$')
# ax1.set_ylabel('j')
# ax1.legend()

# plt.show()




# fig, ax3 = plt.subplots(1)

# x,y = np.linspace(0, x_start+full_x_range - 1, 1000 ) , np.linspace(y_start, y_start+full_y_range -.01, 7) 
# for Y in y: ax3.plot( x, ((np.vectorize( j_int )( x , Y )) - np.vectorize( j )( x , Y )), label=f"j( x , {Y:.02f} )" )
# ax3.set_title("Difference between 15-bit x interpolated and exact j function")
# ax3.set_xlabel("x")
# ax3.set_ylabel('j')
# ax3.legend()

# plt.show()

# print(j_int(100, .44, 2))

# fig, ax1 = plt.subplots(1)

# x = np.linspace(0, .5, 1000 )
# y = .44

# for I in range(len(x_bits)): ax1.plot( x, (abs(np.vectorize( j_int )( x, y, I ) - np.vectorize(j)( x, y ))/ np.vectorize(j)( x, y )), label=f"j( x ," + fr' .44) for {x_bits[I]} x table bits')
# ax1.set_title("Relative Error Between Exact and Interpolated j")
# ax1.set_xlabel(r'$x$')
# ax1.set_ylabel('j')
# ax1.legend()

# plt.show()



# fig, (ax1,ax2) = plt.subplots(2)

# #x = np.linspace(0, 500, 10000 )


# ax1.plot(x, np.vectorize( j )(x, y))
# ax1.plot(x, np.vectorize( j_int )(x, y, 3), linestyle='dashed')
# ax2.plot(x, np.vectorize( j )(x, y) - np.vectorize( j_int )(x, y, 3))

# ax1.set_title(rf"Comparison between exact and interpolated j(x, .44) for 15 x table bits")
# ax1.legend(['Exact', 'interpolation'])
# ax1.set_xlabel('x')
# ax1.set_ylabel('j')

# ax2.set_xlabel('x')
# ax2.set_ylabel('Difference')

# plt.show()




134.02278232574463
